# 02 · Train and evaluate U-Net vs Attention U-Net

Trains both models under identical settings on the data prepared by notebook 01. It then tunes the
binarization threshold on validation, scores the locked test set and draws qualitative comparisons.

**Runtime:** GPU (T4 or L4).

| Run | Model | Loss |
|---|---|---|
| `E1` | U-Net (baseline) | Focal + Dice |
| `E2` | Attention U-Net | Focal + Dice |
| `E0` (optional) | U-Net | BCE, sanity check / imbalance baseline |

Checkpoints are written to Drive after every epoch. After a disconnect, run all cells again.
Finished runs are skipped and interrupted ones resume.

In [ ]:
REPO_URL = "https://github.com/imbishal7/oil-slick-segmentation.git"
BRANCH = "main"
PROJECT_DIR = "/content/drive/MyDrive/oil-slick-segmentation"  # everything persistent lives here

RUN_E0 = False   # optional BCE baseline
EPOCHS = 50      # upper bound; early stopping (patience 10) usually ends sooner
BATCH_SIZE = 8

DATA_DIR = "/content/data"                 # fast local copy of PROJECT_DIR/data
RUNS_DIR = f"{PROJECT_DIR}/runs"           # checkpoints + logs (persistent)
REPORT_DIR = f"{PROJECT_DIR}/reports"

In [ ]:
# Mount Drive, fetch the latest code and install it.
import os, sys
from google.colab import drive

drive.mount("/content/drive")
REPO_DIR = "/content/oil-slick-segmentation"
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull -q --ff-only origin {BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} log -1 --format="code at %h: %s"
!pip install -q -e {REPO_DIR}
sys.path.insert(0, f"{REPO_DIR}/src")  # importable without restarting the runtime
os.makedirs(PROJECT_DIR, exist_ok=True)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!rsync -a --info=progress2 "{PROJECT_DIR}/data/" "{DATA_DIR}/"

In [ ]:
from oilslick import train

def run(experiment: str) -> None:
    train.main([experiment, "--data-dir", DATA_DIR, "--runs-dir", RUNS_DIR,
                "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE), "--resume"])

## E1 · U-Net

In [ ]:
run("E1")

## E2 · Attention U-Net

In [ ]:
run("E2")

## E0 · U-Net + BCE (optional)

In [ ]:
if RUN_E0:
    run("E0")

## Training curves

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd

RUNS = [f"{RUNS_DIR}/{e}" for e in ("E0", "E1", "E2") if os.path.exists(f"{RUNS_DIR}/{e}/best.pt")]

fig, (ax_loss, ax_iou) = plt.subplots(1, 2, figsize=(12, 4))
for run_dir in RUNS:
    h = pd.read_csv(f"{run_dir}/history.csv")
    name = os.path.basename(run_dir)
    ax_loss.plot(h["epoch"], h["train_loss"], label=f"{name} train")
    ax_loss.plot(h["epoch"], h["val_loss"], "--", label=f"{name} val")
    ax_iou.plot(h["epoch"], h["val_iou"], label=name)
ax_loss.set(xlabel="epoch", ylabel="loss", title="Loss")
ax_iou.set(xlabel="epoch", ylabel="oil IoU @ 0.5", title="Validation oil IoU")
ax_loss.legend(); ax_iou.legend()
plt.tight_layout()

## Test-set evaluation

The threshold τ is chosen on validation. The locked test set (150 oil, 150 oil-free, 150 look-alike)
is scored only here.

In [ ]:
from IPython.display import Markdown
from oilslick import evaluate

evaluate.main([*RUNS, "--data-dir", DATA_DIR, "--report-dir", REPORT_DIR])
Markdown(open(f"{REPORT_DIR}/results.md").read())

## Qualitative comparison

The error maps show TP in green, FP in red and FN in blue. The attention maps show the finest
attention-gate coefficients α.

In [ ]:
from glob import glob
from IPython.display import Image, display
from oilslick import visualize

visualize.main([*RUNS, "--data-dir", DATA_DIR, "--out-dir", f"{REPORT_DIR}/figures"])
for path in sorted(glob(f"{REPORT_DIR}/figures/*.png")):
    print(os.path.basename(path))
    display(Image(path))